# Ride Sharing Driver Performance Analysis

## Medallion Architecture

Bronze Layer → Raw Data

Silver Layer → Cleaned Data

Gold Layer → Business KPIs

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [3]:
spark = (
    SparkSession.builder
    .appName("Ride Sharing Analysis")
    .master("local[*]")
    .getOrCreate()
)

print("Spark Started Successfully!")

Spark Started Successfully!


In [4]:
drivers_df = spark.read.csv(
    "../data/drivers.csv",
    header=True,
    inferSchema=True
)

trips_df = spark.read.csv(
    "../data/trips.csv",
    header=True,
    inferSchema=True
)

logs_df = spark.read.csv(
    "../data/trip_logs.csv",
    header=True,
    inferSchema=True
)

In [5]:
drivers_df.show(5)

trips_df.show(5)

logs_df.show(5)

+---------+-------+------+------+
|driver_id|   name|  city|rating|
+---------+-------+------+------+
|        1|Rahul_1| Delhi|   4.6|
|        2|Priya_2|Mumbai|   3.7|
|        3|Rahul_3|  Pune|   3.6|
|        4|Sneha_4| Delhi|   3.5|
|        5|Priya_5|Mumbai|   4.3|
+---------+-------+------+------+
only showing top 5 rows

+-------+---------+---------------+-------------+-----------+-----------+-----------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|      1|       78|        IT Park|      IT Park|        8.6|        0.0|  Cancelled|
|      2|      114|        Airport|         Mall|      17.54|     203.08|  Completed|
|      3|      132|Railway Station|         Mall|      17.27|     213.02|  Completed|
|      4|       58|Railway Station|      IT Park|      20.55|      185.6|  Completed|
|      5|       19|        IT Park|      IT Park|      12.47|     1

In [6]:
drivers_df.printSchema()

trips_df.printSchema()

logs_df.printSchema()

root
 |-- driver_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- rating: double (nullable = true)

root
 |-- trip_id: integer (nullable = true)
 |-- driver_id: integer (nullable = true)
 |-- pickup_location: string (nullable = true)
 |-- drop_location: string (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- trip_status: string (nullable = true)

root
 |-- log_id: integer (nullable = true)
 |-- trip_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- delay_minutes: integer (nullable = true)
 |-- cancellation_flag: integer (nullable = true)



# Bronze Layer

In [7]:
# Save Drivers Data into Bronze Layer

drivers_df.write \
    .mode("overwrite") \
    .parquet("../bronze/drivers")

In [8]:
drivers_df.write.mode("overwrite").parquet(
    r"C:\Users\rishi\OneDrive\Desktop\Final Project\bronze\drivers"
)

print("Drivers Bronze Layer Created!")

Drivers Bronze Layer Created!


In [9]:
trips_df.write.mode("overwrite").parquet(
    r"C:\Users\rishi\OneDrive\Desktop\Final Project\bronze\trips"
)

logs_df.write.mode("overwrite").parquet(
    r"C:\Users\rishi\OneDrive\Desktop\Final Project\bronze\trip_logs"
)

print("Bronze Layer Created Successfully!")

Bronze Layer Created Successfully!


# Data Quality Checks

In [10]:
print("Drivers Records :", drivers_df.count())
print("Trips Records :", trips_df.count())
print("Trip Logs Records :", logs_df.count())

Drivers Records : 150
Trips Records : 150
Trip Logs Records : 150


In [11]:
from pyspark.sql.functions import col, sum, when

print("Drivers Dataset")
drivers_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in drivers_df.columns
]).show()

print("Trips Dataset")
trips_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in trips_df.columns
]).show()

print("Trip Logs Dataset")
logs_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in logs_df.columns
]).show()

Drivers Dataset
+---------+----+----+------+
|driver_id|name|city|rating|
+---------+----+----+------+
|        0|   0|   0|     0|
+---------+----+----+------+

Trips Dataset
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|      0|        0|              0|            0|          0|          0|          0|
+-------+---------+---------------+-------------+-----------+-----------+-----------+

Trip Logs Dataset
+------+-------+----------+--------+-------------+-----------------+
|log_id|trip_id|start_time|end_time|delay_minutes|cancellation_flag|
+------+-------+----------+--------+-------------+-----------------+
|     0|      0|         0|      87|            0|                0|
+------+-------+----------+--------+-------------+-----------------+



# Task 3 - Join Datasets

In [12]:
ride_df = trips_df.join(
    drivers_df,
    on="driver_id",
    how="inner"
)

ride_df = ride_df.join(
    logs_df,
    on="trip_id",
    how="inner"
)

ride_df.show(5)

+-------+---------+---------------+-------------+-----------+-----------+-----------+---------+---------+------+------+-------------------+-------------------+-------------+-----------------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|     name|     city|rating|log_id|         start_time|           end_time|delay_minutes|cancellation_flag|
+-------+---------+---------------+-------------+-----------+-----------+-----------+---------+---------+------+------+-------------------+-------------------+-------------+-----------------+
|      1|       78|        IT Park|      IT Park|        8.6|        0.0|  Cancelled|Anjali_78|    Delhi|   5.0|     1|2025-01-03 01:44:00|               NULL|            0|                1|
|      2|      114|        Airport|         Mall|      17.54|     203.08|  Completed| Neha_114|Bangalore|   3.7|     2|2025-01-02 04:34:00|2025-01-02 04:50:00|           20|                0|
|      3|      132|Railway Station|     

In [13]:
ride_df.printSchema()

root
 |-- trip_id: integer (nullable = true)
 |-- driver_id: integer (nullable = true)
 |-- pickup_location: string (nullable = true)
 |-- drop_location: string (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- log_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- delay_minutes: integer (nullable = true)
 |-- cancellation_flag: integer (nullable = true)



In [14]:
print("Total Records :", ride_df.count())

Total Records : 150


# Task 4 - Data Cleaning
## Check Duplicate Records

In [15]:
print("Total Records :", ride_df.count())

print("Distinct Records :", ride_df.distinct().count())

Total Records : 150
Distinct Records : 150


In [16]:
ride_df.filter(col("distance_km") < 0).show()

+-------+---------+---------------+-------------+-----------+-----------+-----------+----+----+------+------+----------+--------+-------------+-----------------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|name|city|rating|log_id|start_time|end_time|delay_minutes|cancellation_flag|
+-------+---------+---------------+-------------+-----------+-----------+-----------+----+----+------+------+----------+--------+-------------+-----------------+
+-------+---------+---------------+-------------+-----------+-----------+-----------+----+----+------+------+----------+--------+-------------+-----------------+



In [17]:
ride_df.filter(col("fare_amount") < 0).show()

+-------+---------+---------------+-------------+-----------+-----------+-----------+----+----+------+------+----------+--------+-------------+-----------------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|name|city|rating|log_id|start_time|end_time|delay_minutes|cancellation_flag|
+-------+---------+---------------+-------------+-----------+-----------+-----------+----+----+------+------+----------+--------+-------------+-----------------+
+-------+---------+---------------+-------------+-----------+-----------+-----------+----+----+------+------+----------+--------+-------------+-----------------+



In [18]:
ride_df.filter(
    (col("rating") < 1) |
    (col("rating") > 5)
).show()

+-------+---------+---------------+-------------+-----------+-----------+-----------+----+----+------+------+----------+--------+-------------+-----------------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|name|city|rating|log_id|start_time|end_time|delay_minutes|cancellation_flag|
+-------+---------+---------------+-------------+-----------+-----------+-----------+----+----+------+------+----------+--------+-------------+-----------------+
+-------+---------+---------------+-------------+-----------+-----------+-----------+----+----+------+------+----------+--------+-------------+-----------------+



In [19]:
ride_df.filter(col("end_time").isNull()) \
.select(
    "trip_id",
    "trip_status",
    "cancellation_flag",
    "end_time"
).show(20, truncate=False)

+-------+-----------+-----------------+--------+
|trip_id|trip_status|cancellation_flag|end_time|
+-------+-----------+-----------------+--------+
|1      |Cancelled  |1                |NULL    |
|7      |Cancelled  |1                |NULL    |
|9      |Cancelled  |1                |NULL    |
|11     |Cancelled  |1                |NULL    |
|12     |Cancelled  |1                |NULL    |
|16     |Cancelled  |1                |NULL    |
|17     |Cancelled  |1                |NULL    |
|18     |Cancelled  |1                |NULL    |
|21     |Cancelled  |1                |NULL    |
|22     |Cancelled  |1                |NULL    |
|25     |Cancelled  |1                |NULL    |
|26     |Cancelled  |1                |NULL    |
|27     |Cancelled  |1                |NULL    |
|28     |Cancelled  |1                |NULL    |
|31     |Cancelled  |1                |NULL    |
|32     |Cancelled  |1                |NULL    |
|33     |Cancelled  |1                |NULL    |
|34     |Cancelled  

# Task 5 - Create Derived Columns

In [20]:
from pyspark.sql.functions import when, unix_timestamp

ride_df = ride_df.withColumn(
    "trip_duration_minutes",
    when(
        col("end_time").isNotNull(),
        (unix_timestamp("end_time") - unix_timestamp("start_time")) / 60
    ).otherwise(None)
)

ride_df = ride_df.withColumn(
    "completion_flag",
    when(col("trip_status") == "Completed", 1).otherwise(0)
)

In [21]:
ride_df.select(
    "trip_id",
    "trip_status",
    "trip_duration_minutes",
    "completion_flag"
).show(10, truncate=False)

+-------+-----------+---------------------+---------------+
|trip_id|trip_status|trip_duration_minutes|completion_flag|
+-------+-----------+---------------------+---------------+
|1      |Cancelled  |NULL                 |0              |
|2      |Completed  |16.0                 |1              |
|3      |Completed  |34.0                 |1              |
|4      |Completed  |33.0                 |1              |
|5      |Completed  |60.0                 |1              |
|6      |Completed  |14.0                 |1              |
|7      |Cancelled  |NULL                 |0              |
|8      |Completed  |45.0                 |1              |
|9      |Cancelled  |NULL                 |0              |
|10     |Completed  |27.0                 |1              |
+-------+-----------+---------------------+---------------+
only showing top 10 rows



# Task 6 - Filter Invalid Records

In [22]:
silver_df = ride_df.filter(
    (col("distance_km") >= 0)
)

In [23]:
print("Records Before Cleaning :", ride_df.count())
print("Records After Cleaning :", silver_df.count())

Records Before Cleaning : 150
Records After Cleaning : 150


# Task 7 - Silver Layer

In [24]:
silver_df.write \
    .mode("overwrite") \
    .parquet(r"C:\Users\rishi\OneDrive\Desktop\Final Project\silver\ride_data")

print("Silver Layer Created Successfully!")

Silver Layer Created Successfully!


# Task 8 - Driver Performance Metrics

In [25]:
driver_performance_df = (
    silver_df.groupBy(
        "driver_id",
        "name",
        "city"
    )
    .agg(
        count("trip_id").alias("total_trips"),

        sum("completion_flag").alias("completed_trips"),

        round(avg("rating"), 2).alias("avg_rating"),

        round(avg("fare_amount"), 2).alias("avg_fare"),

        round(avg("trip_duration_minutes"), 2).alias("avg_trip_duration")
    )
)

driver_performance_df.show(10, truncate=False)

+---------+----------+---------+-----------+---------------+----------+--------+-----------------+
|driver_id|name      |city     |total_trips|completed_trips|avg_rating|avg_fare|avg_trip_duration|
+---------+----------+---------+-----------+---------------+----------+--------+-----------------+
|26       |Amit_26   |Mumbai   |1          |1              |4.7       |157.9   |53.0             |
|92       |Priya_92  |Hyderabad|1          |1              |3.7       |33.67   |23.0             |
|62       |Anjali_62 |Mumbai   |1          |0              |4.4       |0.0     |NULL             |
|68       |Arjun_68  |Pune     |2          |1              |4.2       |124.5   |7.0              |
|98       |Neha_98   |Bangalore|2          |0              |4.2       |0.0     |NULL             |
|24       |Sneha_24  |Bangalore|1          |1              |5.0       |163.49  |54.0             |
|74       |Priya_74  |Pune     |2          |0              |4.2       |0.0     |NULL             |
|116      

# Task 9 - Cancellation Rate per Driver

In [26]:
from pyspark.sql.functions import round

cancellation_df = (
    silver_df.groupBy("driver_id", "name")
    .agg(
        count("trip_id").alias("total_trips"),
        sum("cancellation_flag").alias("cancelled_trips")
    )
    .withColumn(
        "cancellation_rate",
        round(
            (col("cancelled_trips") / col("total_trips")) * 100,
            2
        )
    )
)

cancellation_df.show(10, truncate=False)

+---------+---------+-----------+---------------+-----------------+
|driver_id|name     |total_trips|cancelled_trips|cancellation_rate|
+---------+---------+-----------+---------------+-----------------+
|8        |Arjun_8  |2          |2              |100.0            |
|75       |Sneha_75 |1          |1              |100.0            |
|114      |Neha_114 |4          |1              |25.0             |
|6        |Amit_6   |1          |1              |100.0            |
|98       |Neha_98  |2          |2              |100.0            |
|11       |Vikas_11 |2          |1              |50.0             |
|2        |Priya_2  |1          |1              |100.0            |
|34       |Priya_34 |1          |1              |100.0            |
|66       |Priya_66 |1          |1              |100.0            |
|103      |Vikas_103|3          |1              |33.33            |
+---------+---------+-----------+---------------+-----------------+
only showing top 10 rows



# Task 10 - High Demand Pickup Locations

In [27]:
pickup_df = (
    silver_df.groupBy("pickup_location")
    .agg(
        count("trip_id").alias("total_trips")
    )
    .orderBy(col("total_trips").desc())
)

pickup_df.show()

+---------------+-----------+
|pickup_location|total_trips|
+---------------+-----------+
|        Airport|         36|
|        IT Park|         34|
|Railway Station|         29|
|           Mall|         26|
|    City Center|         25|
+---------------+-----------+



# Task 11 - Revenue Analysis

In [28]:
revenue_df = (
    silver_df.groupBy("city")
    .agg(
        round(
            sum("fare_amount"),
            2
        ).alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

revenue_df.show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|    Delhi|      2851.14|
|Bangalore|      2535.43|
|   Mumbai|      2286.38|
|     Pune|      1493.44|
|Hyderabad|       924.69|
+---------+-------------+



In [29]:
driver_performance_df.write.mode("overwrite").parquet(
    r"C:\Users\rishi\OneDrive\Desktop\Final Project\gold\driver_performance"
)

cancellation_df.write.mode("overwrite").parquet(
    r"C:\Users\rishi\OneDrive\Desktop\Final Project\gold\cancellation_rate"
)

pickup_df.write.mode("overwrite").parquet(
    r"C:\Users\rishi\OneDrive\Desktop\Final Project\gold\pickup_analysis"
)

revenue_df.write.mode("overwrite").parquet(
    r"C:\Users\rishi\OneDrive\Desktop\Final Project\gold\revenue_analysis"
)

print("Gold Layer Created Successfully!")

Gold Layer Created Successfully!


# Task 13 - Data Validation

In [30]:
print("=" * 40)
print("DATA VALIDATION")
print("=" * 40)

print("Bronze Records :", ride_df.count())
print("Silver Records :", silver_df.count())

print("Driver Performance Records :", driver_performance_df.count())
print("Cancellation Records :", cancellation_df.count())
print("Pickup Analysis Records :", pickup_df.count())
print("Revenue Analysis Records :", revenue_df.count())

DATA VALIDATION
Bronze Records : 150
Silver Records : 150
Driver Performance Records : 99
Cancellation Records : 99
Pickup Analysis Records : 5
Revenue Analysis Records : 5


# Task 14 - Spark Optimization

In [31]:
silver_df.cache()

print("Silver Data Cached Successfully!")

Silver Data Cached Successfully!


In [32]:
silver_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [trip_id#42, driver_id#43, pickup_location#44, drop_location#45, distance_km#46, fare_amount#47, trip_status#48, name#18, city#19, rating#20, log_id#73, start_time#75, end_time#76, delay_minutes#77, cancellation_flag#78, trip_duration_minutes#900, completion_flag#917]
      +- InMemoryRelation [trip_id#42, driver_id#43, pickup_location#44, drop_location#45, distance_km#46, fare_amount#47, trip_status#48, name#18, city#19, rating#20, log_id#73, start_time#75, end_time#76, delay_minutes#77, cancellation_flag#78, trip_duration_minutes#900, completion_flag#917], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=false
               +- Project [trip_id#42, driver_id#43, pickup_location#44, drop_location#45, distance_km#46, fare_amount#47, trip_status#48, name#18, city#19, rating#20, log_id#73, start_time#75, end_time#76, delay_minutes#77, cancellation_flag#78, CASE WH

# Task 15 - Driver Ranking using Window Function

In [33]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

window_spec = Window.orderBy(
    col("completed_trips").desc(),
    col("avg_rating").desc()
)

driver_ranking_df = driver_performance_df.withColumn(
    "driver_rank",
    dense_rank().over(window_spec)
)

driver_ranking_df.show(20, truncate=False)

+---------+---------+---------+-----------+---------------+----------+--------+-----------------+-----------+
|driver_id|name     |city     |total_trips|completed_trips|avg_rating|avg_fare|avg_trip_duration|driver_rank|
+---------+---------+---------+-----------+---------------+----------+--------+-----------------+-----------+
|76       |Priya_76 |Delhi    |3          |3              |4.2       |123.16  |35.67            |1          |
|114      |Neha_114 |Bangalore|4          |3              |3.7       |180.11  |28.33            |2          |
|65       |Amit_65  |Bangalore|3          |3              |3.6       |164.58  |22.33            |3          |
|38       |Sneha_38 |Pune     |2          |2              |5.0       |111.72  |49.5             |4          |
|102      |Vikas_102|Delhi    |2          |2              |4.8       |195.26  |32.0             |5          |
|20       |Vikas_20 |Mumbai   |3          |2              |4.1       |112.22  |23.5             |6          |
|132      

In [34]:
driver_ranking_df.write \
    .mode("overwrite") \
    .parquet(
        r"C:\Users\rishi\OneDrive\Desktop\Final Project\gold\driver_ranking"
    )

print("Driver Ranking Saved Successfully!")

Driver Ranking Saved Successfully!
